In [2]:
import pandas as pd
import os
import io
import re
from io import BytesIO
from ftplib import FTP

In [3]:
def connect_ftp(server, username, password):
    """
    Descripcion : Conecta al servidor FTP con las credenciales proporcionadas.
    Parámetros:
    -server (str): Dirección del servidor FTP.
    -username (str): Nombre de usuario para la autenticación.
    -password (str): Contraseña para la autenticación.
    returns:
    -ftp (FTP): Objeto FTP conectado al servidor.
    """
    ftp = FTP(server)
    ftp.login(user=username, passwd=password)
    return ftp

def close_ftp(ftp):
    """
    Descripcion : Cierra la conexión FTP.
    Parámetros:
    -ftp (FTP): Objeto FTP que se desea cerrar.
    returns:
    None
    """
    ftp.quit()

def read_csv_from_ftp(ftp, remote_file_path):
    """
    Descripcion : Lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parámetros:
    -ftp (FTP): Objeto FTP conectado al servidor.
    -remote_file_path (str): Ruta del archivo CSV en el servidor FTP.
    returns:
    -df (DataFrame): DataFrame de pandas que contiene los datos del archivo CSV.
    """
    with BytesIO() as bio:
        ftp.retrbinary(f'RETR {remote_file_path}', bio.write)
        bio.seek(0)
        df = pd.read_csv(bio)
    return df

def CargarFTP(df,direccion):
    """
    Descripcion : Carga un DataFrame de pandas a un servidor FTP como un archivo CSV.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que se desea cargar.
    -direccion (str): Ruta remota donde se guardará el archivo CSV en el servidor FTP.
    returns:
    None
    """
    csv_buffer = io.BytesIO() 
    df.to_csv(csv_buffer, index=False, encoding='utf-8')
    csv_buffer.seek(0) 
    ftp = connect_ftp(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password')) 
    remote_path = direccion 
    ftp.storbinary(f'STOR {remote_path}', csv_buffer) 
    close_ftp(ftp) 

def upload_csv_to_ftp(df, path):
    """
    Descripcion : Sube un DataFrame de pandas a un servidor FTP como un archivo CSV.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que se desea subir.
    -path (str): Ruta remota donde se guardará el archivo CSV en el servidor FTP.
    returns:
    None
    """
    csv_buffer = io.BytesIO()
    df.to_csv(csv_buffer, index=False, encoding='utf-8')
    csv_buffer.seek(0)
    ftp = connect_ftp(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    ftp.storbinary(f'STOR {path}', csv_buffer)
    close_ftp(ftp)
    
def Read_data(namearc):
    """
    Descripcion : Lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parámetros:
    -namearc (str): Nombre del archivo CSV que se desea leer.
    returns:
    -df (DataFrame): DataFrame de pandas que contiene los datos del archivo CSV.
    """
    ftp = connect_ftp(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    df = read_csv_from_ftp(ftp, f'/pruebas/csv/Recodificacion/{namearc}')
    close_ftp(ftp)
    return df 

def Read_data2(namearc):
    """
    Descripcion : Lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parámetros:
    -namearc (str): Nombre del archivo CSV que se desea leer.
    returns:
    -df (DataFrame): DataFrame de pandas que contiene los datos del archivo CSV.
    """
    ftp = connect_ftp(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    df = read_csv_from_ftp(ftp, f'/pruebas/csv/Sankey/{namearc}')
    close_ftp(ftp)
    return df   

In [4]:
def Columns_to_dummis(df):
    """
    Descripcion : Genera una lista de columnas para el DataFrame que contiene las columnas de sustancias, orden de consumo y edad de inicio.
    Parámetros:
    -df (DataFrame): DataFrame de pandas del cual se extraerán las columnas.
    returns:
    -list_col (list): Lista de columnas que contiene 'FolioId' y las columnas de sustancias, orden de consumo y edad de inicio.
    """
    list1 = [col for col in df.columns if col.startswith('SustanciaId')]
    list2 = [col for col in df.columns if col.startswith('OrdenConsumo')]
    list3 = [col for col in df.columns if col.startswith('EdadInicio')]
    list_col = ['FolioId'] + list1 + list2 + list3 
    return list_col

def gen_valueof_column():
    """
    Descripcion : Genera un diccionario que mapea los nombres de las sustancias a sus respectivas columnas.
    returns:
    -dict: Diccionario que contiene los nombres de las sustancias como claves y sus respectivas columnas como valores.
    """
    return {
        'Tabaco': 'Tabaco',
        'Alcohol': 'Alcohol',
        'Marihuana': 'Marihuana',
        'Cocaína': 'Cocaína',
        'Metanfetaminas': 'Metanfetaminas',
        'Opioides': 'Opioides',
        'Inhalables': 'Inhalables',
        'Alucinógenos': 'Alucinógenos',
        'Medicamentos': 'Medicamentos',
        'Nuevas Sustancias Psicoactivas': 'Nuevas Sustancias Psicoactivas',
        'Otras Sustancias': 'Otras Sustancias',
    }

def orden(row):
    """
    Descripcion : Calcula el orden de consumo basado en las edades de inicio de consumo de sustancias.
    Parámetros:
    -row (Series): Fila del DataFrame que contiene las edades de inicio de consumo de sustancias.
    returns:
    -orden (list): Lista que contiene el orden de consumo de las sustancias basado en las edades de inicio.
    """
    Edades = [row[f'EdadInicio{i}'] for i in range(1, 30)]
    indices_valores = [(i, e) for i, e in enumerate(Edades) if pd.notna(e)]
    valores_no_nulos_ordenados = sorted(indices_valores, key=lambda x: x[1])
    orden = [None] * len(Edades)
    for orden_idx, (original_idx, _) in enumerate(valores_no_nulos_ordenados):
        orden[original_idx] = orden_idx + 1
    return orden

def OrdenPorEdad (df):
    """
    Descripcion : Calcula el orden de consumo de sustancias basado en las edades de inicio de consumo.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las edades de inicio de consumo de sustancias.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas que indican el orden de consumo de las sustancias.
    """
    list_colOrden = [f'OrdenConsumo{i}' for i in range(1, 30)]
    df[list_colOrden] = df.apply(orden, axis=1, result_type="expand")
    return df

def ModOrden(df):
    """
    Descripcion : Modifica el orden de consumo de sustancias para que las sustancias con el mismo ID tengan el mismo orden.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con las columnas de orden de consumo modificadas.
    """
    sustancia_columns = [col for col in df.columns if col.startswith('SustanciaId')]
    orden_columns = [col for col in df.columns if col.startswith('OrdenConsumo')]


    for index, row in df.iterrows():

        min_values = {}
        for sustancia_col, orden_col in zip(sustancia_columns, orden_columns):
            sustancia = row[sustancia_col]
            orden = row[orden_col]
            

            if pd.notna(sustancia) and pd.notna(orden):
                if sustancia not in min_values:
                    min_values[sustancia] = orden
                else:
                    min_values[sustancia] = min(min_values[sustancia], orden)
        
        for sustancia_col, orden_col in zip(sustancia_columns, orden_columns):
            sustancia = row[sustancia_col]
            if pd.notna(sustancia):
                df.at[index, orden_col] = min_values.get(sustancia, row[orden_col])
            else:
                df.at[index, orden_col] = row[orden_col]  
    return df

def Recodificacion(df):
    """
    Descripcion : Recode las sustancias en columnas binarias y asigna el orden de consumo correspondiente.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas binarias para cada sustancia y su orden de consumo.
    """
    list_colSustancia = [col for col in df.columns if col.startswith('SustanciaId')]
    list_colOrden = [col for col in df.columns if col.startswith('OrdenConsumo')]

    value_to_column = gen_valueof_column()
    for sust, nueva_col in value_to_column.items():
        for sustancia_col, orden_col in zip(list_colSustancia, list_colOrden):
            for ind, val in df[sustancia_col].items():
                if val == sust:
                    df.at[ind, nueva_col] = 1
                    df.at[ind, nueva_col + 'Orden'] = df.at[ind, orden_col]
    return df

def Metodo_1(df):
    """
    Descripcion : Asigna la primera sustancia y su consumo basado en el orden de consumo.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas para la primera sustancia y su consumo.
    """
    df['PrimerConsumo'] = None
    df['PrimeraSustancia'] = None
    gen_valueof = gen_valueof_column()
    for key, value in gen_valueof.items():
        for ind, val in df[key+'Orden'].items():
            if val == 1:
                df.at[ind, 'PrimeraSustancia'] = key
                df.at[ind, 'PrimerConsumo'] = val
    return df

def Metodo_2(df):
    """
    Descripcion : Asigna la segunda sustancia y su consumo basado en el orden de consumo.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas para la segunda sustancia y su consumo.
    """
    df['SegundaSustancia'] = None
    df['SegundoConsumo'] = None
    gen_valueof = gen_valueof_column()
    for key, value in gen_valueof.items():
        for ind, val in df[key+'Orden'].items():
            if val == 2:
                df.at[ind, 'SegundaSustancia'] = key
                df.at[ind, 'SegundoConsumo'] = val
    return df

def Metodo_3(df):
    """
    Descripcion : Asigna la tercera sustancia y su consumo basado en el orden de consumo.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas para la tercera sustancia y su consumo.
    """
    df['TercerSustancia'] = None
    df['TercerConsumo'] = None
    gen_valueof = gen_valueof_column()
    for key, value in gen_valueof.items():
        for ind, val in df[key+'Orden'].items():
            if val == 3:
                df.at[ind, 'TercerSustancia'] = key
                df.at[ind, 'TercerConsumo'] = val
    return df

def Metodo_4(df):
    """
    Descripcion : Asigna la cuarta sustancia y su consumo basado en el orden de consumo.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas para la cuarta sustancia y su consumo.
    """
    df['CuartaSustancia'] = None
    df['CuartoConsumo'] = None
    gen_valueof = gen_valueof_column()
    for key, value in gen_valueof.items():
        for ind, val in df[key+'Orden'].items():
            if val == 4:
                df.at[ind, 'CuartaSustancia'] = key
                df.at[ind, 'CuartoConsumo'] = val
    return df

def Metodo_5(df):
    """
    Descripcion : Asigna la quinta sustancia y su consumo basado en el orden de consumo.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df (DataFrame): DataFrame de pandas con nuevas columnas para la quinta sustancia y su consumo.
    """
    df['QuintaSustancia'] = None
    df['QuintoConsumo'] = None
    gen_valueof = gen_valueof_column()
    for key, value in gen_valueof.items():
        for ind, val in df[key+'Orden'].items():
            if val == 5:
                df.at[ind, 'QuintaSustancia'] = key
                df.at[ind, 'QuintoConsumo'] = val
    return df

def sankey(df):
    """
    Descripcion : Crea un DataFrame para un diagrama de Sankey basado en las sustancias consumidas y su orden.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las columnas de sustancias y orden de consumo.
    returns:
    -df_sankey (DataFrame): DataFrame de pandas que contiene las relaciones entre las sustancias y sus consumos para el diagrama de Sankey.
    """
    sustancias = pd.concat([df['PrimeraSustancia'], df['SegundaSustancia'], df['TercerSustancia'], df['CuartaSustancia'], df['QuintaSustancia']]).unique()
    sankey_data = []
    for sust in sustancias:
        dfaux = df[df['PrimeraSustancia'] == sust]
        for sust2 in sustancias:
            value = len(dfaux[dfaux['SegundaSustancia'] == sust2])
            if value > 0:
                sankey_data.append({'from': sust + '1', 'to': sust2 + '2', 'value': value})
                dfaux2 = dfaux[dfaux['SegundaSustancia'] == sust2]
                for sust3 in sustancias:
                    value2 = len(dfaux2[dfaux2['TercerSustancia'] == sust3])
                    if value2 > 0:
                        sankey_data.append({'from': sust2 + '2', 'to': sust3 + '3', 'value': value2})
                        dfaux3 = dfaux2[dfaux2['TercerSustancia'] == sust3]
                        for sust4 in sustancias:
                            value3 = len(dfaux3[dfaux3['CuartaSustancia'] == sust4])
                            if value3 > 0:
                                sankey_data.append({'from': sust3 + '3', 'to': sust4 + '4', 'value': value3})
                                dfaux4 = dfaux3[dfaux3['CuartaSustancia'] == sust4]
                                for sust5 in sustancias:
                                    value4 = len(dfaux4[dfaux4['QuintaSustancia'] == sust5])
                                    if value4 > 0:
                                        sankey_data.append({'from': sust4 + '4', 'to': sust5 + '5', 'value': value4})
    
    df_sankey = pd.DataFrame(sankey_data)
    return df_sankey

def modder(dfsankey):
    """
    Descripcion : Modifica el DataFrame de Sankey para combinar las relaciones entre sustancias y sumar sus valores.
    Parámetros:
    -dfsankey (DataFrame): DataFrame de pandas que contiene las relaciones entre las sustancias y sus consumos para el diagrama de Sankey.
    returns:
    -dfsankey (DataFrame): DataFrame de pandas modificado que contiene las relaciones combinadas entre las sustancias y sus consumos.
    """
    dfsankey['Comb'] = dfsankey['from'] + dfsankey['to']
    combinaciones = dfsankey['Comb'].unique()

    for combi in combinaciones:
        filtered_df = dfsankey[dfsankey['Comb'] == combi]
        new_value = filtered_df['value'].sum()
        dfsankey.loc[dfsankey['Comb'] == combi, 'value'] = new_value

    dfsankey = dfsankey.drop_duplicates(subset = ['Comb'])
    dfsankey = dfsankey.drop(columns = 'Comb')
    dfsankey.insert(0,'Sust','sust')
    return dfsankey

def sankeyhistorico(df):
    """
    Descripcion : Combina el DataFrame de Sankey actual con un DataFrame histórico y agrega una columna de sostenibilidad.
    Parámetros:
    -df (DataFrame): DataFrame de pandas que contiene las relaciones entre las sustancias y sus consumos para el diagrama de Sankey.
    returns:
    -df_agg (DataFrame): DataFrame de pandas combinado que contiene las relaciones entre las sustancias y sus consumos, incluyendo datos históricos y una columna de sostenibilidad.
    """
    df2 =Read_data2('Sankey_Historico.csv')
    df_combined = pd.concat([df, df2], ignore_index=True)
    df_agg = df_combined.groupby(['from', 'to'], as_index=False)['value'].sum()
    df_agg['Sust'] = 'sust'
    df_agg = df_agg[['Sust', 'from', 'to', 'value']]
    return df_agg

def main ():
    try:
        df = pd.DataFrame()
        df = Read_data('EntrevistaInicialDrogasLegalesIlegales.csv')
        list_col = Columns_to_dummis(df)
        df = df[list_col].copy()
        df = OrdenPorEdad(df)
        df = ModOrden(df)
        df = Recodificacion(df)
        df = Metodo_1(df)
        df = Metodo_2(df)
        df = Metodo_3(df)
        df = Metodo_4(df)
        df = Metodo_5(df)
        df = Metodo_1(df)
        dfsankey = sankey(df)
        dfsankey = modder(dfsankey)
        dfsankey = sankeyhistorico(dfsankey)
        upload_csv_to_ftp(dfsankey, r'/pruebas/csv/Sankey/sankey.csv')
    except Exception as e:
        print(f"6)_Sankey - Error: {e}")
    return dfsankey

In [5]:
if __name__ == "__main__":
    df = main()

C:\Users\franc\AppData\Local\Temp\ipykernel_27260\3246316661.py:37: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,322,330,344,345,346,347,348,349,366,367,370,371,372,373,374,375,424,425) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)
